# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HemapavaniDontula/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### 1. Two paper findings + my methodology questions

**Finding 1:** The research found that some content items show stronger signals of decline than others. The decline label is derived from the defined trend-based outcome. My methodology question is whether the validation design is strong enough to show that this relationship generalizes beyond the data used to develop the model.

**Finding 2:** The model provided a useful ranking of content items compared with the rule-based baseline. The label comes from the defined decline outcome, while the model uses the feature set available before the outcome period. My methodology question is whether the comparison remains valid when clients are separated during validation and when the model is evaluated only on unseen data.

Overall, these findings are useful as directional evidence, but they should not be interpreted as proof of causal relationships or guaranteed future predictions.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: methodology checks

print("Label column:", "is_declining_label")
print("Validation design should use grouped or time-aware splitting.")
print("trend_pct and trend_direction must not be features.")
print("client_id must be used for grouping/splitting, not as a model feature.")

Label column: is_declining_label
Validation design should use grouped or time-aware splitting.
trend_pct and trend_direction must not be features.
client_id must be used for grouping/splitting, not as a model feature.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### 2. My model under an honest split (before/after)

I re-evaluated the Week-5 model using a more honest validation design. The purpose was to check whether the model's performance remains useful when validation is separated by client or by time rather than relying only on a random split.

The baseline and model were evaluated on the same validation observations. This makes the comparison fair because both methods receive the same held-out data.

The honest validation result is treated as the more reliable estimate of generalization performance than the original development result.


In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import spearmanr

RANDOM_STATE = 42

repo_path = Path("/content/flyrank-ml")

if not repo_path.exists():
    !git clone --depth 1 https://github.com/HemapavaniDontula/flyrank-ml.git /content/flyrank-ml

os.chdir(repo_path)

data_path = repo_path / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, val_idx = next(
    splitter.split(
        df,
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("Train clients:", train_df["client_id"].nunique())
print("Validation clients:", val_df["client_id"].nunique())

overlap = set(train_df["client_id"]) & set(val_df["client_id"])
print("Client overlap:", len(overlap))

In [11]:
train_df["decline_severity"] = -pd.to_numeric(
    train_df["trend_pct"],
    errors="coerce"
)

val_df["decline_severity"] = -pd.to_numeric(
    val_df["trend_pct"],
    errors="coerce"
)

train_df = train_df.dropna(
    subset=["decline_severity"]
).copy()

val_df = val_df.dropna(
    subset=["decline_severity"]
).copy()

feature_cols = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "position_tier",
    "avg_position",
    "ctr"
]

X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()

y_train = train_df["decline_severity"].copy()
y_val = val_df["decline_severity"].copy()

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "position_tier"
]

numeric_features = [
    col for col in feature_cols
    if col not in categorical_features
]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (21133, 17)
X_val: (5479, 17)
y_train: (21133,)
y_val: (5479,)


In [12]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]
)

print("Random Forest pipeline created.")

Random Forest pipeline created.


In [13]:
rf_pipeline.fit(X_train, y_train)

model_pred = rf_pipeline.predict(X_val)

print("Model trained successfully.")
print("Predictions:", len(model_pred))

Model trained successfully.
Predictions: 5479


In [14]:
baseline_train = train_df.copy()
baseline_val = val_df.copy()

stale_cutoff = baseline_train[
    "days_since_last_update"
].quantile(2/3)

position_ctr_median = (
    baseline_train
    .groupby("position_tier")["ctr"]
    .median()
)

baseline_val["_stale_signal"] = (
    baseline_val["days_since_last_update"] >= stale_cutoff
).astype(int)

baseline_val["_position_ctr_median"] = (
    baseline_val["position_tier"]
    .map(position_ctr_median)
)

baseline_val["_ctr_signal"] = (
    baseline_val["ctr"] <
    baseline_val["_position_ctr_median"]
).astype(int)

baseline_val["_position_signal"] = (
    baseline_val["position_tier"].isin(
        ["top_3", "page_1"]
    )
).astype(int)

baseline_val["baseline_score"] = (
    2 * baseline_val["_stale_signal"]
    + 2 * baseline_val["_ctr_signal"]
    + baseline_val["_position_signal"]
)

print(baseline_val["baseline_score"].head())

0     0
1     0
5     3
13    2
19    1
Name: baseline_score, dtype: int64


In [15]:
comparison_df = pd.DataFrame({
    "actual_decline_severity": y_val.values,
    "baseline_score": baseline_val["baseline_score"].values,
    "model_prediction": model_pred
})

baseline_spearman = spearmanr(
    comparison_df["baseline_score"],
    comparison_df["actual_decline_severity"]
).statistic

model_spearman = spearmanr(
    comparison_df["model_prediction"],
    comparison_df["actual_decline_severity"]
).statistic

comparison_table = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Spearman_rank_correlation": [
        baseline_spearman,
        model_spearman
    ]
})

print(comparison_table.to_string(index=False))

         Method  Spearman_rank_correlation
Week-4 baseline                   0.186973
  Random Forest                   0.124757


In [5]:
# Check the important variables needed for ML-09
variables_to_check = [
    "X_train", "X_val",
    "y_train", "y_val",
    "baseline_val"
]

for v in variables_to_check:
    print(v, "->", v in globals())

X_train -> False
X_val -> False
y_train -> False
y_val -> False
baseline_val -> False


In [3]:
# Find variables that may contain the trained model
[x for x in globals().keys()
 if any(k in x.lower() for k in ["model", "rf", "xgb", "forest", "regressor", "classifier"])]

[]

In [ ]:
# Basic comparison

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

model_mae = mean_absolute_error(y_val, model_pred)
model_rmse = np.sqrt(mean_squared_error(y_val, model_pred))
model_r2 = r2_score(y_val, model_pred)

baseline_pred = baseline_val["baseline_score"].values

baseline_mae = mean_absolute_error(y_val, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_val, baseline_pred))
baseline_r2 = r2_score(y_val, baseline_pred)

print("MODEL")
print("MAE :", model_mae)
print("RMSE:", model_rmse)
print("R2  :", model_r2)

print("\nBASELINE")
print("MAE :", baseline_mae)
print("RMSE:", baseline_rmse)
print("R2  :", baseline_r2)

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### 3. Leakage audit

I checked the final feature set for target leakage and identifier leakage.

The target `decline_severity` is derived from `trend_pct`, so `trend_pct` must not be included as a model feature. `trend_direction` is also excluded because it is derived from the same trend information used to define the outcome.

I also excluded `is_declining_label`, because it directly represents the decline outcome.

`content_id` and `client_id` are identifiers rather than predictive features. `client_id` is used for grouped validation, but it is not supplied to the model.

The final feature set contains only the selected content and search signals used before evaluating the observed outcome. The leakage audit therefore checks both the feature list and the validation data for known leakage or identifier columns.

The results are treated as a validation check, not as proof that all possible forms of leakage have been eliminated.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage audit for the final feature set

leakage_columns = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "decline_severity",
    "content_id",
    "client_id"
]

print("LEAKAGE / IDENTIFIER AUDIT")
print("=" * 50)

found_in_features = [
    col for col in leakage_columns
    if col in feature_cols
]

if found_in_features:
    print("WARNING: These columns are present in feature_cols:")
    for col in found_in_features:
        print("-", col)
else:
    print("PASS: No known leakage or identifier columns are in feature_cols.")

print("\nFinal feature count:", len(feature_cols))
print("\nFinal model features:")
for col in feature_cols:
    print("-", col)

LEAKAGE / IDENTIFIER AUDIT
PASS: No known leakage or identifier columns are in feature_cols.

Final feature count: 17

Final model features:
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- content_age_days
- age_tier_order
- days_since_last_update
- freshness_tier
- word_count_tier
- char_count_tier
- position_tier
- avg_position
- ctr


In [17]:
print("\nACTUAL MODEL INPUT AUDIT")
print("=" * 50)

bad_train = [
    col for col in leakage_columns
    if col in X_train.columns
]

bad_val = [
    col for col in leakage_columns
    if col in X_val.columns
]

print("Bad columns in X_train:", bad_train)
print("Bad columns in X_val:", bad_val)

if not bad_train and not bad_val:
    print("PASS: X_train and X_val contain no known leakage/identifier columns.")
else:
    print("WARNING: Leakage or identifier columns detected.")


ACTUAL MODEL INPUT AUDIT
Bad columns in X_train: []
Bad columns in X_val: []
PASS: X_train and X_val contain no known leakage/identifier columns.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### 4. Claim rewrite

**Original strong claim:**

The Random Forest model can predict which content items will decline and identify the pages that should be fixed first.

**Safer claim:**

On the evaluated validation data, the Random Forest model showed a measurable relationship between its ranking scores and observed decline severity. The model was compared with the Week-4 rule-based baseline on the same held-out client groups.

These results are directional and should be used as decision-support for prioritizing content for further review. They should not be interpreted as proof that the model will reliably predict future decline or that the identified signals cause the observed changes.


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("CLAIM AUDIT")
print("=" * 40)

print("Model evaluation: Random Forest")
print("Validation design: Grouped by client_id")
print("Outcome: Observed decline severity (-trend_pct)")

print("\nClaim language:")
print("- Observed")
print("- Measured")
print("- Directional")
print("- Decision-support")

print("\nThe model should not be interpreted as:")
print("- Proof of causation")
print("- Guaranteed future prediction")
print("- A guaranteed rule for content changes")

CLAIM AUDIT
Model evaluation: Random Forest
Validation design: Grouped by client_id
Outcome: Observed decline severity (-trend_pct)

Claim language:
- Observed
- Measured
- Directional
- Decision-support

The model should not be interpreted as:
- Proof of causation
- Guaranteed future prediction
- A guaranteed rule for content changes


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.